# Computer-use Agent：从截图坐标到可验证动作闭环

**面试问题：screenshot–ground–act–observe 循环怎样处理坐标、陈旧画面、审批和幂等？**

## 回答主线

先把真实请求和资源合同摆出来，用最简单的方案建立成本或正确性基线，再手写核心控制逻辑并展示完整事件、指标和失败修正。断言只在最后保护少量关键不变量，前面的可见输入、过程和结果才是学习主体。

## 真实案例

财务 Agent 需要在网页发票系统里把订单 A-7842 的邮寄地址改为上海，然后点击“保存草稿”，不能直接提交开票。案例给出截图尺寸、DOM 可访问元素、页面版本和用户授权；对比硬编码像素与语义 grounding，展示缩放坐标、动作 digest、审批、权威回读，以及页面弹窗导致旧截图坐标失效的安全阻断。

### 输入预览：页面快照、DOM 元素和用户目标

In [1]:
import hashlib  # 导入摘要函数把审批绑定到精确动作参数。
import json  # 导入稳定 JSON 序列化生成动作 digest。

snapshot = {"page_id": "invoice-A7842", "version": 17, "image_size": (1440, 900), "device_scale": 1.25}  # 定义截图对应的权威页面版本与坐标尺度。
elements = [  # 构造从 DOM/可访问树提取的语义元素。
    {"ref": "e-address", "role": "textbox", "name": "邮寄地址", "bbox": (420, 260, 980, 310), "value": "北京市朝阳区"},  # 地址输入框包含当前权威值。
    {"ref": "e-draft", "role": "button", "name": "保存草稿", "bbox": (980, 760, 1120, 810), "enabled": True},  # 低风险保存草稿按钮。
    {"ref": "e-submit", "role": "button", "name": "提交开票", "bbox": (1140, 760, 1300, 810), "enabled": True},  # 高风险正式提交按钮需要额外审批。
]  # 完成可操作元素快照。
goal = {"order": "A-7842", "new_address": "上海市浦东新区世纪大道 100 号", "allowed_action": "保存草稿"}  # 构造用户明确授权的目标和动作边界。
print("页面快照：", snapshot)  # 展示所有坐标和动作都必须绑定的版本。
print("用户目标：", goal)  # 展示新地址和不得正式提交的授权边界。
for element in elements:  # 逐项展示语义 grounding 输入。
    print(element)  # 显示 ref、role、name、bbox 和当前状态。

页面快照： {'page_id': 'invoice-A7842', 'version': 17, 'image_size': (1440, 900), 'device_scale': 1.25}
用户目标： {'order': 'A-7842', 'new_address': '上海市浦东新区世纪大道 100 号', 'allowed_action': '保存草稿'}
{'ref': 'e-address', 'role': 'textbox', 'name': '邮寄地址', 'bbox': (420, 260, 980, 310), 'value': '北京市朝阳区'}
{'ref': 'e-draft', 'role': 'button', 'name': '保存草稿', 'bbox': (980, 760, 1120, 810), 'enabled': True}
{'ref': 'e-submit', 'role': 'button', 'name': '提交开票', 'bbox': (1140, 760, 1300, 810), 'enabled': True}


## Baseline 基线：直接复用录制脚本中的像素坐标

In [2]:
recorded_click = (1190, 785)  # 构造旧自动化脚本记录的固定像素坐标。

def element_at(point, current_elements):  # 根据坐标查找当前页面真正命中的元素。
    x_coordinate, y_coordinate = point  # 解包待点击的截图坐标。
    for element in current_elements:  # 遍历当前快照中的全部可操作元素。
        left, top, right, bottom = element["bbox"]  # 读取元素包围框。
        if left <= x_coordinate <= right and top <= y_coordinate <= bottom:  # 判断点击是否落在该元素内部。
            return element  # 返回真实命中的元素。
    return None  # 没有元素命中时返回空结果。

baseline_hit = element_at(recorded_click, elements)  # 在当前页面解释旧脚本坐标。
print("旧坐标点击位置：", recorded_click)  # 展示基线只依赖像素而没有语义目标。
print("实际命中元素：", baseline_hit)  # 暴露坐标落在“提交开票”而不是“保存草稿”。

旧坐标点击位置： (1190, 785)
实际命中元素： {'ref': 'e-submit', 'role': 'button', 'name': '提交开票', 'bbox': (1140, 760, 1300, 810), 'enabled': True}


### 核心实现：语义 grounding、缩放换算与动作计划

In [3]:
def find_element(role, name, current_elements):  # 按 role 和可访问名称做确定性语义 grounding。
    matches = [element for element in current_elements if element["role"] == role and element["name"] == name]  # 过滤精确匹配元素。
    if len(matches) != 1:  # 零个或多个匹配都不能安全执行动作。
        raise LookupError(f"元素匹配数量异常：{len(matches)}")  # 返回明确 grounding 错误。
    return matches[0]  # 返回唯一可验证元素。

def center_in_device_pixels(element, scale):  # 把 CSS/截图 bbox 中心换算到设备像素坐标。
    left, top, right, bottom = element["bbox"]  # 读取语义元素包围框。
    return (round((left + right) / 2 * scale), round((top + bottom) / 2 * scale))  # 应用设备 scale 得到实际输入坐标。

address_element = find_element("textbox", "邮寄地址", elements)  # 定位需要修改的地址字段。
draft_element = find_element("button", goal["allowed_action"], elements)  # 定位用户允许的保存草稿按钮。
action_plan = [  # 构造绑定 page/version/ref 的两步动作计划。
    {"type": "fill", "page": snapshot["page_id"], "version": snapshot["version"], "ref": address_element["ref"], "value": goal["new_address"]},  # 第一步填写精确新地址。
    {"type": "click", "page": snapshot["page_id"], "version": snapshot["version"], "ref": draft_element["ref"], "point": center_in_device_pixels(draft_element, snapshot["device_scale"])},  # 第二步点击语义匹配的草稿按钮。
]  # 完成可审计动作计划。
print("语义 grounding 后的动作计划：")  # 输出确定性动作序列。
for action in action_plan:  # 逐步展示参数、版本和 ref。
    print(action)  # 显示动作不再依赖无语义的固定像素。

语义 grounding 后的动作计划：
{'type': 'fill', 'page': 'invoice-A7842', 'version': 17, 'ref': 'e-address', 'value': '上海市浦东新区世纪大道 100 号'}
{'type': 'click', 'page': 'invoice-A7842', 'version': 17, 'ref': 'e-draft', 'point': (1312, 981)}


### 审批绑定、幂等提交与权威回读

In [4]:
def digest_action(action):  # 对动作参数生成稳定 SHA-256 摘要。
    payload = json.dumps(action, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 使用确定性 JSON 防止字段顺序改变摘要。
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()  # 返回绑定精确参数和版本的摘要。

approval = {"ticket": "approve-901", "allowed_digest": digest_action(action_plan[-1]), "risk": "draft-only", "used": False}  # 构造只允许保存草稿动作的一次性审批。
page_state = {"address": "北京市朝阳区", "draft_saved": False, "submitted": False, "version": 17}  # 构造页面后端权威状态。
ledger = {}  # 创建基于 invocation id 的幂等结果账本。

def execute_plan(plan, current_state, approval_ticket, invocation_id):  # 执行带版本、审批和幂等保护的动作计划。
    if invocation_id in ledger:  # 重试相同调用时直接返回已提交结果。
        return ledger[invocation_id]  # 避免重复点击产生二次副作用。
    if any(action["version"] != current_state["version"] for action in plan):  # 检查所有动作仍绑定当前页面版本。
        raise RuntimeError("页面版本已变化，必须重新观察")  # 阻止陈旧截图上的动作。
    if digest_action(plan[-1]) != approval_ticket["allowed_digest"] or approval_ticket["used"]:  # 验证审批精确绑定且尚未消费。
        raise PermissionError("审批不匹配或已使用")  # 阻止参数替换和审批重放。
    current_state["address"] = plan[0]["value"]  # 执行输入框填写的确定性效果。
    current_state["draft_saved"] = True  # 执行保存草稿而非正式提交。
    current_state["version"] += 1  # 页面成功提交后推进权威版本。
    approval_ticket["used"] = True  # 消费一次性审批防止重复动作。
    ledger[invocation_id] = {"status": "success", "state": current_state.copy()}  # 保存可重放的权威结果。
    return ledger[invocation_id]  # 返回执行结果供 Agent 观察。

result = execute_plan(action_plan, page_state, approval, "invoke-A7842-draft")  # 执行经过 grounding 与审批的安全计划。
retry_result = execute_plan(action_plan, page_state, approval, "invoke-A7842-draft")  # 模拟网络超时后的相同调用重试。
print("首次执行结果：", result)  # 展示地址、草稿状态和新页面版本。
print("幂等重试结果：", retry_result)  # 展示重试读取同一账本而没有再次点击。

首次执行结果： {'status': 'success', 'state': {'address': '上海市浦东新区世纪大道 100 号', 'draft_saved': True, 'submitted': False, 'version': 18}}
幂等重试结果： {'status': 'success', 'state': {'address': '上海市浦东新区世纪大道 100 号', 'draft_saved': True, 'submitted': False, 'version': 18}}


## 结果解读：观察—动作—回读闭环

In [5]:
observed_diff = {"address": ("北京市朝阳区", result["state"]["address"]), "draft_saved": (False, result["state"]["draft_saved"]), "submitted": (False, result["state"]["submitted"]), "version": (17, result["state"]["version"])}  # 构造执行前后权威状态差异。
print("权威状态 diff：")  # 输出功能完成证据标题。
for field, change in observed_diff.items():  # 逐字段展示预期和非预期副作用。
    print(f"{field:<12} {change[0]} -> {change[1]}")  # 显示地址和草稿变化且正式提交仍为假。
print("解读：工具返回 success 不是完成证据；只有重新读取权威状态并核对 diff，Agent 才能声称任务完成。")  # 强调后置条件验证。

权威状态 diff：
address      北京市朝阳区 -> 上海市浦东新区世纪大道 100 号
draft_saved  False -> True
submitted    False -> False
version      17 -> 18
解读：工具返回 success 不是完成证据；只有重新读取权威状态并核对 diff，Agent 才能声称任务完成。


## 失败案例：弹窗使旧快照和 ref 全部陈旧

In [6]:
stale_state = {"address": "北京市朝阳区", "draft_saved": False, "submitted": False, "version": 18}  # 模拟合规弹窗出现后页面版本推进。
stale_error = None  # 初始化陈旧动作错误结果。
fresh_approval = {"ticket": "approve-902", "allowed_digest": digest_action(action_plan[-1]), "risk": "draft-only", "used": False}  # 构造尚未使用但仍绑定旧版本的审批。
try:  # 尝试在新页面上执行旧快照计划。
    execute_plan(action_plan, stale_state, fresh_approval, "invoke-stale")  # 发起应被版本门禁拒绝的动作。
except RuntimeError as error:  # 捕获预期陈旧页面错误。
    stale_error = str(error)  # 保存错误供重新观察逻辑使用。
print("陈旧动作结果：", stale_error)  # 展示系统没有盲点旧坐标或复用旧 ref。
print("修正路径：重新截图/获取 DOM -> 重新 grounding -> 重新计算动作 digest -> 必要时重新审批。")  # 给出真实 Computer-use loop 的恢复步骤。

陈旧动作结果： 页面版本已变化，必须重新观察
修正路径：重新截图/获取 DOM -> 重新 grounding -> 重新计算动作 digest -> 必要时重新审批。


### 生产边界

In [7]:
trace = {"page": snapshot["page_id"], "observed_version": snapshot["version"], "action_refs": [action["ref"] for action in action_plan], "approval": approval["ticket"], "invocation": "invoke-A7842-draft", "post_version": result["state"]["version"], "submitted": result["state"]["submitted"]}  # 汇总不包含截图 PII 的低敏执行 trace。
print("安全执行 trace：", trace)  # 展示调试时需要的版本、ref、审批和后置状态。
print("生产替换点：还需浏览器隔离、OCR/DOM 融合、坐标校准、敏感区域遮罩、下载治理、人工接管和任务级评测。")  # 明确教学页面模型与真实 GUI Agent 的差距。

安全执行 trace： {'page': 'invoice-A7842', 'observed_version': 17, 'action_refs': ['e-address', 'e-draft'], 'approval': 'approve-901', 'invocation': 'invoke-A7842-draft', 'post_version': 18, 'submitted': False}
生产替换点：还需浏览器隔离、OCR/DOM 融合、坐标校准、敏感区域遮罩、下载治理、人工接管和任务级评测。


## 回归测试：只保护语义目标、权限和后置条件

In [8]:
assert baseline_hit["name"] == "提交开票"  # 验证固定坐标基线确实暴露了高风险误点。
assert action_plan[-1]["ref"] == "e-draft"  # 验证语义 grounding 选择用户允许的保存草稿按钮。
assert result == retry_result and len(ledger) == 1  # 验证相同 invocation 重试没有重复副作用。
assert result["state"]["draft_saved"] and not result["state"]["submitted"]  # 验证只保存草稿且没有越权正式提交。
assert stale_error is not None  # 验证页面版本变化会阻断旧动作计划。
print("回归测试通过：误点探针、语义 grounding、幂等、授权边界和陈旧页面门禁均成立。")  # 用少量断言总结安全闭环。

回归测试通过：误点探针、语义 grounding、幂等、授权边界和陈旧页面门禁均成立。
